In [1]:
import torch

pred = torch.zeros(1, 2, requires_grad=True)
target = torch.tensor([[0.035, 0.48]])  # narrow column, wide column

loss = (pred - target).abs().mean()  # L1
loss.backward()

print("loss:", loss.item())
print("grad:", pred.grad)


loss: 0.2574999928474426
grad: tensor([[-0.5000, -0.5000]])


In [2]:
(0.035 + 0.48) / 2

0.2575

In [19]:
0.48 / 0.035

13.714285714285712

```-
              ⎧  x    if x > 0
        |x| = ⎨
              ⎩ -x    if x < 0

  d|x|/dx =   +1   for x > 0     (derivative of  x)
              -1   for x < 0     (derivative of -x)
              undefined at x = 0 (left slope -1, right slope +1)

  so for  L = g(|x|),  chain rule:

        ∂L/∂x = ∂L/∂|x| · d|x|/dx = grad_output · sign(x)


```


In [11]:
x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 3.0], requires_grad=True)
x.abs().sum().backward()
print(x.grad)
torch.equal(x.grad, x.sign())

tensor([-1., -1.,  0.,  1.,  1.])


True

```-
  L1: L = |e|     →  dL/de = sign(e)   constant ±1, even when e → 0
  L2: L = e²      →  dL/de = 2e        shrinks as e → 0

```


In [ ]:
x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 3.0], requires_grad=True)
x.square().sum().backward()
print(x.grad)


tensor([-4., -1.,  0.,  1.,  6.])


In [16]:
import torch

pred = torch.zeros(1, 2, requires_grad=True)
target = torch.tensor([[0.035, 0.48]], requires_grad=True)  # narrow column, wide column

loss = (pred - target).abs().mean()  # L1
loss.backward()

print("loss:", loss.item())
print("grad:", pred.grad)
print(f"{target.grad=}")

loss: 0.2574999928474426
grad: tensor([[-0.5000, -0.5000]])
target.grad=tensor([[0.5000, 0.5000]])


In [22]:
pred = torch.zeros(1, 2, requires_grad=True)
target = torch.tensor([[0.035, 0.48]], requires_grad=True)  # narrow column, wide column
loss = ((pred - target) ** 2).mean()  # L2 / MSE
loss.backward()

print("loss:", loss.item())
print("grad:", pred.grad)
print("ratio:", (pred.grad[0, 1] / pred.grad[0, 0]).item())
print(f"{target.grad=}")

loss: 0.11581249535083771
grad: tensor([[-0.0350, -0.4800]])
ratio: 13.714284896850586
target.grad=tensor([[0.0350, 0.4800]])


In [24]:
spread = torch.tensor([0.035, 0.48])  # natural std of each column
err = torch.tensor([0.01, 0.01])  # the same absolute error in both

print("absolute error :", err)
print("as % of spread :", (100 * err / spread).round(decimals=1).tolist())


absolute error : tensor([0.0100, 0.0100])
as % of spread : [28.600000381469727, 2.0999999046325684]


In [ ]:
spread = torch.tensor([0.035, 0.48])

pred = torch.zeros(1, 2, requires_grad=True)
target = torch.tensor([[0.035, 0.48]], requires_grad=True)

loss = (pred - target / spread).abs().mean()  # both sides normalised
loss.backward()

print("grad in normalised space:", pred.grad)
print("effective grad per radian:", pred.grad / spread)


grad in normalised space: tensor([[-0.5000, -0.5000]])
effective grad per radian: tensor([[-14.2857,  -1.0417]])


In [ ]:
x = torch.tensor([[0.3, 30.0]])  # i1 small scale, i2 large scale
w = torch.zeros(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

loss = ((x @ w + b) - torch.tensor([[1.0]])).abs().mean()
loss.backward()

print("dL/dy :", -0.5)
print("dL/dw :", w.grad.flatten().tolist())
print("ratio :", (w.grad[1] / w.grad[0]).item())


dL/dy : -0.5
dL/dw : [-0.30000001192092896, -30.0]
ratio : 99.99999237060547


In [28]:
x = torch.tensor([[0.3, 30.0]])  # i1 small scale, i2 large scale
x = (x - x.mean()) / x.std()
w = torch.zeros(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

loss = ((x @ w + b) - torch.tensor([[1.0]])).abs().mean()
loss.backward()

print("dL/dy :", -0.5)
print("dL/dw :", w.grad.flatten().tolist())
print("ratio :", (w.grad[1] / w.grad[0]).item())


dL/dy : -0.5
dL/dw : [0.7071067094802856, -0.7071067690849304]
ratio : -1.0000001192092896


In [84]:
import torch

torch.manual_seed(0)

N = 200
i1 = 0.3 + 0.1 * torch.randn(N)  # scale ~0.1 around 0.3
i2 = 30.0 + 10.0 * torch.randn(N)  # scale ~10  around 30
X = torch.stack([i1, i2], 1)

TRUE_W, TRUE_B = torch.tensor([3.0, 0.02]), 0.5
y = (X * TRUE_W).sum(1, keepdim=True) + TRUE_B  # ground truth, no noise


def train(Xi, lr, steps=5000):
    w = torch.zeros(2, 1, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    hist = []
    for s in range(steps):
        loss = ((Xi @ w + b) - y).abs().mean()
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
            w.grad.zero_()
            b.grad.zero_()
        hist.append(loss.item())
    return w.detach().flatten(), b.detach().item(), hist


mu, sd = X.mean(0), X.std(0)
Xn = (X - mu) / sd

for lr in [1e-3, 1e-2]:
    wr, br, hr = train(X, lr)
    wn, bn, hn = train(Xn, lr)
    print(f"lr={lr}")
    print(
        f"  RAW   loss {hr[0]:.3f} -> {hr[-1]:7.4f}   w = {[round(v, 4) for v in wr.tolist()]}   b = {br:.4f}"
    )
    print(
        f"  NORM  loss {hn[0]:.3f} -> {hn[-1]:7.4f}   w = {[round(v, 4) for v in (wn / sd).tolist()]}  (rescaled to raw units)   b = {bn:.4f}"
    )
print(
    f"  TRUE                          w = {[round(v, 4) for v in TRUE_W.tolist()]}   b = {TRUE_B:.4f}"
)


lr=0.001
  RAW   loss 1.985 ->  0.4405   w = [0.4475, 0.0247]   b = 0.8135
  NORM  loss 1.985 ->  0.0007   w = [2.9999, 0.02]  (rescaled to raw units)   b = 1.9850
lr=0.01
  RAW   loss 1.985 ->  7.9334   w = [0.1903, 0.0212]   b = 0.4753
  NORM  loss 1.985 ->  0.0036   w = [2.9897, 0.0199]  (rescaled to raw units)   b = 1.9911
  TRUE                          w = [3.0, 0.02]   b = 0.5000


In [ ]:
torch.manual_seed(0)
N = 400
X = torch.randn(N, 4)  # inputs already well-scaled: isolate the target effect
W_TRUE = torch.randn(4, 2)
W_TRUE[:, 0] *= 0.035 / W_TRUE[:, 0].std()  # column 0: narrow
W_TRUE[:, 1] *= 0.48 / W_TRUE[:, 1].std()  # column 1: wide
Y = X @ W_TRUE
print("target std per column:", Y.std(0).round(decimals=4).tolist())


def train(norm_targets, steps=4000, lr=1e-2):
    torch.manual_seed(1)
    net = torch.nn.Linear(4, 2)
    opt = torch.optim.AdamW(net.parameters(), lr=lr)
    sd = Y.std(0) if norm_targets else torch.ones(2)
    for s in range(steps):
        loss = ((net(X) - Y / sd).abs()).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    with torch.no_grad():
        return ((net(X) * sd) - Y).abs().mean(0)  # always scored in raw units


for name, nt in [("raw targets", False), ("normalised targets", True)]:
    err = train(nt)
    rel = 100 * err / Y.std(0)
    print(
        f"{name:20s} abs err {[f'{v:.4f}' for v in err.tolist()]}   rel err {[f'{v:.1f}%' for v in rel.tolist()]}"
    )


target std per column: [0.06469999998807907, 0.9839000105857849]
raw targets          abs err ['0.0010', '0.0011']   rel err ['1.6%', '0.1%']
normalised targets   abs err ['0.0001', '0.0009']   rel err ['0.1%', '0.1%']
